In [ ]:
import os
import zipfile
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Embedding, LSTM, Add
)

# ==========================================================
# METHOD 1: BUILT-IN FASHION-MNIST DATASET
# Images are loaded into NumPy arrays, not an image folder.
# This is a demonstration dataset and has labels, not captions.
# ==========================================================

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Use a smaller subset to keep the demonstration manageable.
x_train = x_train[:1000].astype("float32") / 255.0
y_train = y_train[:1000]

x_test = x_test[:100].astype("float32") / 255.0
y_test = y_test[:100]

class_names = [
    "t-shirt", "trouser", "pullover", "dress", "coat",
    "sandal", "shirt", "sneaker", "bag", "ankle boot"
]

# Add the grayscale channel dimension expected by the CNN.
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# Generate simple demonstration captions from the class labels.
# These are synthetic captions, not original human-written captions.

captions = []

for label in y_train:
    name = class_names[label]
    article = "an" if name[0] in "aeiou" else "a"

    templates = [
        "this is " + article + " " + name,
        "look at " + article + " " + name
    ]

    captions.append("startseq " + random.choice(templates) + " endseq")


# ==========================================================
# METHOD 2: CUSTOM IMAGE FOLDER + CSV
# If using your own dataset, comment out METHOD 1 above.
# Uncomment this section and update the paths.
#
# DATASET_PATH = r"C:\path\to\dataset"
# IMAGE_FOLDER = os.path.join(DATASET_PATH, "images")
# CAPTION_FILE = os.path.join(DATASET_PATH, "captions.csv")
#
# captions_df = pd.read_csv(CAPTION_FILE)
#
# images = []
# captions = []
#
# for _, row in captions_df.iterrows():
#     image_path = os.path.join(IMAGE_FOLDER, row["image_name"])
#     if os.path.exists(image_path):
#         img = load_img(image_path, target_size=(128, 128))
#         images.append(img_to_array(img) / 255.0)
#         captions.append("startseq " + str(row["caption"]) + " endseq")
#
# x_train = np.array(images)
# y_train = None
#
# # For a proper experiment, split the images and captions into
# # training and testing sets together, keeping their order aligned.
# # Set x_test and y_test to the held-out images and labels/captions.
#
#
# ==========================================================
# METHOD 3: CUSTOM ZIP FILE
# Extract the ZIP first, then use the folder-loading approach.
#
# ZIP_PATH = r"C:\path\to\dataset.zip"
# EXTRACT_PATH = r"C:\path\to\extracted_dataset"
#
# with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
#     zip_ref.extractall(EXTRACT_PATH)
#
# # Then set DATASET_PATH = EXTRACT_PATH in METHOD 2.
# ==========================================================

print("Training image shape:", x_train.shape)
print("Number of captions:", len(captions))
# The model processes word IDs instead of raw text.
# The Tokenizer builds a vocabulary from the training captions.

tokenizer = Tokenizer()
tokenizer.fit_on_texts(captions)

vocab_size = len(tokenizer.word_index) + 1
sequences = tokenizer.texts_to_sequences(captions)

max_len = max(len(seq) for seq in sequences)

print("Vocabulary size:", vocab_size)
print("Maximum caption length:", max_len)

# Create next-word prediction examples.
# Example:
# Input: "startseq this"
# Target: "is"

X_img = []
X_seq = []
Y = []

for i, seq in enumerate(sequences):
    for j in range(1, len(seq)):
        X_img.append(x_train[i])
        X_seq.append(seq[:j])
        Y.append(seq[j])

# Pad all partial captions to the same length.
X_seq = pad_sequences(X_seq, maxlen=max_len, padding="pre")

X_img = np.array(X_img, dtype="float32")
X_seq = np.array(X_seq)
Y = np.array(Y)

print("Image inputs:", X_img.shape)
print("Caption inputs:", X_seq.shape)
print("Next-word targets:", Y.shape)
# CNN branch: extracts features from the image.
image_input = Input(shape=x_train.shape[1:], name="image_input")

x = Conv2D(32, (3, 3), activation="relu", padding="same")(image_input)
x = MaxPooling2D((2, 2))(x)

x = Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = MaxPooling2D((2, 2))(x)

x = GlobalAveragePooling2D()(x)
image_features = Dense(64, activation="relu")(x)


# LSTM branch: processes the caption words generated so far.
caption_input = Input(shape=(max_len,), name="caption_input")

x = Embedding(
    input_dim=vocab_size,
    output_dim=64
)(caption_input)

text_features = LSTM(64)(x)


# Combine image and text features.
merged = Add()([image_features, text_features])
merged = Dense(64, activation="relu")(merged)

# Predict the next word in the caption.
output = Dense(vocab_size, activation="softmax")(merged)

model = Model(
    inputs=[image_input, caption_input],
    outputs=output
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
# The model learns to predict the next word from the image
# and the words already present in the partial caption.

history = model.fit(
    [X_img, X_seq],
    Y,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)
# Visualize training progress.
# This displays the graph without saving it as a PNG file.

plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("CNN + LSTM Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()
# Create a mapping from numerical word IDs back to words.
index_to_word = {
    index: word
    for word, index in tokenizer.word_index.items()
}

# Generate captions for six test images.
for i in range(min(6, len(x_test))):

    img = x_test[i]
    generated_caption = "startseq"

    for j in range(max_len):

        # Convert the current caption into word IDs.
        seq = tokenizer.texts_to_sequences([generated_caption])[0]
        seq = pad_sequences([seq], maxlen=max_len, padding="pre")

        # Predict the next word.
        prediction = model.predict(
            [img[np.newaxis, ...], seq],
            verbose=0
        )

        next_word_id = np.argmax(prediction)

        # Stop if padding or an unknown token is predicted.
        if next_word_id == 0:
            break

        next_word = index_to_word.get(next_word_id, "")

        # Stop when the caption reaches its end token.
        if next_word == "endseq" or next_word == "":
            break

        generated_caption += " " + next_word

    display_caption = generated_caption.replace("startseq", "").strip()

    # Show the image and generated caption.
    plt.figure(figsize=(4, 4))
    plt.imshow(img.squeeze(), cmap="gray")
    plt.title(
        "Actual: " + class_names[y_test[i]] +
        "\nGenerated: " + display_caption
    )
    plt.axis("off")
    plt.show()


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Example captions: ['startseq a photo of a bag endseq', 'startseq a photo of a bag endseq', 'startseq an image containing a sneaker endseq']
Vocab size: 27, Max caption length: 9
Training pairs: 24819


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 28, 28,    │        320 │ image_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 14, 14,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 14, 14,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 7, 7, 64)  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ caption_input       │ (None, 9)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ max_pooling2d_1[… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 9, 128)    │      3,456 │ caption_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      8,320 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 9, 128)    │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 9)         │          0 │ caption_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 128)       │    131,584 │ dropout_1[0][0],  │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128)       │          0 │ dropout[0][0],    │
│                     │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     16,512 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 27)        │      3,483 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 182,171 (711.61 KB)

 Trainable params: 182,171 (711.61 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 43s 110ms/step - accuracy: 0.6308 - loss: 1.1129 - val_accuracy: 0.7365 - val_loss: 0.6369
Epoch 2/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 36s 104ms/step - accuracy: 0.7449 - loss: 0.5992 - val_accuracy: 0.7595 - val_loss: 0.5620
Epoch 3/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 41s 105ms/step - accuracy: 0.7581 - loss: 0.5422 - val_accuracy: 0.7695 - val_loss: 0.5014
Epoch 4/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 40s 102ms/step - accuracy: 0.7775 - loss: 0.4858 - val_accuracy: 0.7917 - val_loss: 0.4636
Epoch 5/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 41s 102ms/step - accuracy: 0.7847 - loss: 0.4682 - val_accuracy: 0.7804 - val_loss: 0.4733
Epoch 6/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 41s 102ms/step - accuracy: 0.7952 - loss: 0.4423 - val_accuracy: 0.7865 - val_loss: 0.4324
Epoch 7/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 42s 105ms/step - accuracy: 0.8000 - loss: 0.4348 - val_accuracy: 0.8114 - val_loss: 0.4181
Epoch 8/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 40s 103ms/step - accuracy: 0.8071 - loss: 0